# Configuration

In [1]:
file_name = ''
speakers_expected = 5

In [2]:
# Jupyter stuff
# See https://ipython.org/ipython-doc/3/config/extensions/autoreload.html
%load_ext autoreload
%autoreload 2

# Configure Jupyter to display high-resolution figures
%config InlineBackend.figure_format = "retina"

# Actual config
import logging
import os
from pathlib import Path, PurePath

from dotenv import load_dotenv
from lib.config_functions import *

load_dotenv()  # Load environment variables

path_base = Path.cwd()
path_media = Path(path_base).joinpath('media')
path_output = Path(path_base).joinpath('output')
path_report_assets = Path(path_base).joinpath('report_assets')
path_prompt_templates = Path(path_base).joinpath('prompt_templates')
path_file = Path(path_media).joinpath(file_name)

# Create the working dir for the given file
file_name_noext = PurePath(path_file).stem  # Remove the extension if any
path_work_dir = Path(path_output).joinpath(file_name_noext)
path_work_dir.mkdir(exist_ok=True)

# Create assemblyai dir
path_transcript_dir = path_work_dir.joinpath('assemblyai')
path_transcript_dir.mkdir(exist_ok=True)

# Create llm dir
path_llm_dir = path_work_dir.joinpath('llm')
path_llm_dir.mkdir(exist_ok=True)

# Create report dir
path_reports_dir = path_work_dir.joinpath('reports')
path_reports_dir.mkdir(exist_ok=True)

# Configure logger
logger = logging.getLogger('root')
logger.handlers = []  # To allow to run this cell multiple times without creating more loggers
logger_add_handlers_file_stream(
    logger=logger, level=logging.DEBUG, filename=str(path_work_dir.joinpath('log.txt'))
)

In [3]:
# TODO: DELETEME After some time
# Hack to move files from root to directory and folder file name change
import os

# Rename folder
if os.path.exists(path_work_dir.joinpath('assemblyai_transcript')):
    os.rename(path_work_dir.joinpath('assemblyai_transcript'), path_work_dir.joinpath('assemblyai'))

if os.path.exists(path_work_dir.joinpath('conversations_summaries.json')):
    os.rename(path_work_dir.joinpath('conversations_summaries.json'), path_llm_dir.joinpath('OLD.conversations_summaries.json'))
if os.path.exists(path_work_dir.joinpath('speaker_mapping.json')):
    os.rename(path_work_dir.joinpath('speaker_mapping.json'), path_llm_dir.joinpath('OLD.speaker_mapping.json'))

# Move chord
if os.path.exists(path_work_dir.joinpath('chord.png')):
    os.rename(path_work_dir.joinpath('chord.png'), path_reports_dir.joinpath('chord.png'))


# Obtaining audio transcript

In [ ]:
import lib.assemblyai_helper_functions as ahf

path_response_upload_json = path_transcript_dir.joinpath('response_upload.json')
path_response_create_json = path_transcript_dir.joinpath('response_create.json')
path_response_transcript_json = path_transcript_dir.joinpath('response_transcript.json')
print(path_response_transcript_json)

transcript = ahf.get_audio_transcript(
    path_file,
    path_response_upload_json=path_response_upload_json,
    path_response_create_json=path_response_create_json,
    path_response_transcript_json=path_response_transcript_json,
    speakers_expected=speakers_expected,
    force_upload=False,
    force_regen=False,
)

# Speaker Mapping and Summarization

In [5]:
import lib.llm_functions as llmfn

convo_summarizer = llmfn.ConversationSummarizer(
    path_save_dir=path_llm_dir, path_prompt_templates=path_prompt_templates
)

In [ ]:
speaker_map = convo_summarizer.speaker_map(
    utterances=transcript['utterances'], speakers_expected=speakers_expected, force_regen=False
)
# speaker_map

# Data, Obtaining and computing

## DataFrames creation from transcript

In [7]:
import pandas as pd

df_utter = pd.DataFrame.from_dict(transcript['utterances'])

# Remove speaker 'UNK', from df_utter
UNK_index = df_utter.query('speaker == "UNK"').index
df_utter.drop(index=UNK_index, inplace=True)

# Add word count column
df_utter['word_count'] = df_utter['text'].apply(lambda x: len(x.split(' ')))

# Add target (the next person to speak as a proxy to whom one talked)
df_utter['target'] = df_utter.loc[:, ['speaker']].shift(periods=-1)

# Add previous (the previous person to speak as a proxy to whom one responds)
df_utter['previous'] = df_utter.loc[:, ['speaker']].shift(periods=1)

# Add words per second and per minute
df_utter['talk_time_sec'] = (df_utter['end'] - df_utter['start']) / 1000
df_utter['words_per_sec'] = df_utter['word_count'] / df_utter['talk_time_sec']
df_utter['words_per_min'] = df_utter['words_per_sec'] * 60

df_sentiment = pd.DataFrame.from_dict(transcript['sentiment_analysis_results'])
# Remove speaker 'UNK', from df_sentiment
UNK_index = df_sentiment.query('speaker == "UNK"').index
df_sentiment.drop(index=UNK_index, inplace=True)
# Adding column that maps sentiment to int values
# See the following to understand why the `with ...` statement
# https://github.com/pandas-dev/pandas/issues/57734#issuecomment-2099489978
with pd.option_context('future.no_silent_downcasting', True):
    df_sentiment['sentiment_value'] = (
        df_sentiment['sentiment']
        .astype('object')
        .replace({'NEGATIVE': -1, 'NEUTRAL': 0, 'POSITIVE': 1})
    )

In [8]:
# df_utter.head(6)
# df_sentiment.head(6)
# df_sentiment.query('sentiment == "NEGATIVE" ')

## Speaker stats

In [ ]:
import nltk

nltk.download('punkt_tab')  # Needed for `nltk.word_tokenize`
nltk.download('averaged_perceptron_tagger_eng')  # Needed for `nltk.pos_tag`


def idea_count(text: str):
    tokens = nltk.word_tokenize(text)
    words = [word for word, tag in nltk.pos_tag(tokens) if tag == 'MD']
    return len(words)


def unique_words(df: pd.DataFrame):
    # IMPORTANT: Function to be applied to a complete DataFrame, returns a new DataFrame
    # see: https://stackoverflow.com/a/35208597/1071459
    words = df['text'].split()
    unique_words = set(words)
    df['unique_words_count'] = len(unique_words)
    df['unique_words_pct'] = len(unique_words) / len(words)
    return df


# Create a df with all text by the same speaker is joined and with the word count
df_speaker_stats = df_utter.groupby(['speaker']).agg(
    {
        'text': lambda x: ' '.join(x),
        'word_count': lambda x: x.sum(),
        'talk_time_sec': lambda x: x.sum(),
        'words_per_min': lambda x: x.mean(),
    }
)
df_speaker_stats['word_count_pct'] = (
    100 * df_speaker_stats['word_count'] / df_speaker_stats['word_count'].sum()
)
df_speaker_stats['talk_time_pct'] = (
    100 * df_speaker_stats['talk_time_sec'] / df_speaker_stats['talk_time_sec'].sum()
)
df_speaker_stats['question_count'] = df_speaker_stats['text'].apply(lambda x: x.count('?'))
df_speaker_stats['idea_count'] = df_speaker_stats['text'].apply(idea_count)
df_speaker_stats = df_speaker_stats.apply(unique_words, axis=1)

# Adding interaction_count
_interaction_count = df_utter['speaker'].value_counts().rename('interaction_count')
df_speaker_stats = pd.merge(df_speaker_stats, _interaction_count, left_index=True, right_index=True)

# Adding action items count
map_speaker_to_action_count = {}
for speaker in speaker_map:
    map_speaker_to_action_count[speaker] = len(speaker_map[speaker]['personal_action_items'])
df_speaker_stats = pd.merge(df_speaker_stats, pd.Series(map_speaker_to_action_count, name='action_items_count'), left_on='speaker', right_index=True)

# At this point we copy the speaker column to have a backup of the original names
# and we change the index to be named 'named'
df_speaker_stats['speaker'] = df_speaker_stats.index
df_speaker_stats.index.name = 'name'

# Sort columns
df_speaker_stats = df_speaker_stats[
    [
        'speaker',
        'text',
        'word_count',
        'word_count_pct',
        'words_per_min',
        'talk_time_sec',
        'talk_time_pct',
        'question_count',
        'idea_count',
        'unique_words_count',
        'unique_words_pct',
        'interaction_count',
        'action_items_count',
    ]
]
# df_speaker_stats

In [10]:
# df_speaker_stats
# df_utter

## Mapping id to real name

In [11]:
# df_speaker_stats.reset_index().set_index('name')
map_speaker_to_name = {key: speaker_map[key]['name'] for key in speaker_map}

df_sentiment['name'] = df_sentiment['speaker'].replace(map_speaker_to_name)

df_utter['name'] = df_utter['speaker'].replace(map_speaker_to_name)
df_utter['name_target'] = df_utter['target'].replace(map_speaker_to_name)
df_utter['name_previous'] = df_utter['previous'].replace(map_speaker_to_name)

df_speaker_stats.reset_index(drop=True,inplace=True)
df_speaker_stats['name'] = df_speaker_stats['speaker'].replace(map_speaker_to_name)
df_speaker_stats = df_speaker_stats.set_index('name')

In [12]:
# df_sentiment.head(4)
# df_utter.head(4)
# df_speaker_stats

## Conversation (script)

In [13]:
conversation_script = ''
for utter in df_utter.index:
    name = df_utter.loc[utter]['name']
    text = df_utter.loc[utter]['text']
    conversation_script += f'* {name}: "{text}"' + '\n'

In [14]:
# # Show conversation_script in nice readable way
# import textwrap

# for line in conversation_script.splitlines():
#     print(textwrap.fill(line, width=100))

## Title and Summaries (long and short)

In [ ]:
summaries = convo_summarizer.conversation_summaries(
    conversation_script=conversation_script,
    speakers_expected=speakers_expected,
    force_regen=False,
)
# summaries

In [ ]:
from IPython.display import display, Markdown

print('summary word count:', len(summaries['conversation_summary_long'].split()))
display(Markdown(summaries['conversation_summary_long']))

In [ ]:
from IPython.display import display, Markdown

print('summary_one_sentence word count:', len(summaries['conversation_summary_short'].split()))
display(Markdown(summaries['conversation_summary_short']))

## Speaker interactions

In [ ]:
# Interactions from name to name_target
# interesting because it shows how many interactions happened
df_interactions_next = pd.crosstab(df_utter['name'], df_utter['name_target'], margins=True)
df_interactions_next

In [ ]:
# Interactions from name to name_previous
# interesting because it shows how many interactions happened
df_interactions_prev = pd.crosstab(df_utter['name'], df_utter['name_previous'], margins=True)
df_interactions_prev

In [ ]:
# Words spoken from name to name_target
# interesting because it shows how the interactions went
df_interactions_next_wrd_cnt = pd.crosstab(
    index=df_utter['name'],
    columns=df_utter['name_target'],
    values=df_utter['word_count'],
    aggfunc='sum',
    margins=True,
).fillna(0).astype('int')
df_interactions_next_wrd_cnt

In [ ]:
# Words spoken from name to name_previous
# interesting because it shows how the interactions went
df_interactions_prev_wrd_cnt = pd.crosstab(
    index=df_utter['name'],
    columns=df_utter['name_previous'],
    values=df_utter['word_count'],
    aggfunc='sum',
    margins=True,
).fillna(0).astype('int')
df_interactions_prev_wrd_cnt

# Report components

## Compute Gini Coefficient, various possibilities

In [22]:
# # Test Gini coef
# import lib.helper_functions as hf
# hf.gini([30/16]*16 + [40/8]*8 + [50/4]*4 + [60/2]*2 + [70]) # Should be 0.6038709677419355

In [23]:
import lib.helper_functions as hf

# Gini with the number of interventions per participants
#
# (take the 'All' column of df_interventions_next without the last [total] column)
list_interventions = df_interactions_next['All'][:-1].to_list()
gini_interventions = hf.gini(list_interventions)
gini_reverse_interventions = 1 - hf.gini(list_interventions)

# Gini with the number of interactions between every participant
#
# # IMPORTANT: df_interactions_next and df_interactions_prev produce the same results
# as the contain the same numbers only transposed
_df = df_interactions_next.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df.shape[0]
for i in range(size):
    # print(_df.iat[i, i])
    _df.iat[i, i] = pd.NA
list_interactions = _df.melt()['value'].dropna().to_list()
gini_interactions = hf.gini(list_interactions)
gini_reverse_interactions = 1 - hf.gini(list_interactions)

# Gini with the number of words in every interactions between every participant to the next person
_df = df_interactions_next_wrd_cnt.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df.shape[0]
for i in range(size):
    # print(_df.iat[i, i])
    _df.iat[i, i] = pd.NA
list_interactions_next_wrd_cnt = _df.melt()['value'].dropna().to_list()
gini_interactions_next_wrd_cnt = hf.gini(list_interactions_next_wrd_cnt)
gini_reverse_interactions_next_wrd_cnt = 1 - hf.gini(list_interactions_next_wrd_cnt)

# Gini with the number of words in every interactions between every participant to the previous person
_df = df_interactions_prev_wrd_cnt.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df.shape[0]
for i in range(size):
    # print(_df.iat[i, i])
    _df.iat[i, i] = pd.NA
list_interactions_prev_wrd_cnt = _df.melt()['value'].dropna().to_list()
gini_interactions_prev_wrd_cnt = hf.gini(list_interactions_prev_wrd_cnt)
gini_reverse_interactions_prev_wrd_cnt = 1 - hf.gini(list_interactions_prev_wrd_cnt)

# Gini with the number of words in every interactions between every participant to the previous person
# and also to the next person (adding both lists)
_df_prev = df_interactions_prev_wrd_cnt.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df_prev.shape[0]
for i in range(size):
    _df_prev.iat[i, i] = pd.NA
_df_next = df_interactions_next_wrd_cnt.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df_next.shape[0]
for i in range(size):
    _df_next.iat[i, i] = pd.NA
list_interactions_both_wrd_cnt = _df_prev.melt()['value'].dropna().to_list() + _df_next.melt()['value'].dropna().to_list()
gini_interactions_both_wrd_cnt = hf.gini(list_interactions_both_wrd_cnt)
gini_reverse_interactions_both_wrd_cnt = 1 - hf.gini(list_interactions_both_wrd_cnt)

### Number of interventions by each participan

In [ ]:
print('Gini, Number of interventions by each participan')
print(
    'gini:',
    round(gini_interventions, 2),
    '[1-gini]:',
    round(gini_reverse_interventions, 2),
)
fig = hf.lorenz_plot(list_interventions)

### Number of every interaction between every participant

In [ ]:
print('Gini, Number of every interaction between every participant')
print(
    'gini:',
    round(gini_interactions, 2),
    '[1-gini]:',
    round(gini_reverse_interactions, 2),
)
fig = hf.lorenz_plot(list_interactions)

### Number of words between every participant, taking to the next person

In [ ]:
print('Gini, Number of words between every participant, taking to the next person')
print(
    'gini:',
    round(gini_interactions_next_wrd_cnt, 2),
    '[1-gini]:',
    round(gini_reverse_interactions_next_wrd_cnt, 2),
)
fig = hf.lorenz_plot(list_interactions_next_wrd_cnt)

### Number of words between every participant, taking to the previous person

In [ ]:
print('Gini, Number of words between every participant, taking to the previous person')
print(
    'gini:',
    round(gini_interactions_prev_wrd_cnt, 2),
    '[1-gini]:',
    round(gini_reverse_interactions_prev_wrd_cnt, 2),
)
fig = hf.lorenz_plot(list_interactions_prev_wrd_cnt)

### Number of words between every participant, taking to the previous person and to the next person

In [ ]:
print('Gini, Number of words between every participant, taking to the previous person and talking to the next person')
print(
    'gini:',
    round(gini_interactions_both_wrd_cnt, 2),
    '[1-gini]:',
    round(gini_reverse_interactions_both_wrd_cnt, 2),
)
fig = hf.lorenz_plot(list_interactions_both_wrd_cnt)

## Nice Speaker Stats

In [29]:
nice_stats_col_name = {
    'action_items_count': 'Action Items',
    'idea_count': 'Idea Generation',
    'question_count': 'Questions Asked',
    'talk_time_pct': 'Speaking Time (%)',
    'interaction_count': 'Total Interactions',
    'unique_words_pct': 'Unique Words % (Idea Density)',
    'word_count': 'Word Count',
    'words_per_min': 'Words Per Minute (WPM)',
    # 'speaker': 'speaker',
    # 'text': 'text',
    # 'word_count_pct': 'word_count_pct',
    # 'talk_time_sec': 'talk_time_sec',
    # 'unique_words_count': 'unique_words_count',
}

In [ ]:
nice_stats_data = (
    df_speaker_stats[nice_stats_col_name.keys()]
    .rename(columns=nice_stats_col_name)
    .sort_index(axis=0)
    # .query('name != "Mel" and name != "Jason"')
)
nice_stats_data['Unique Words % (Idea Density)'] = nice_stats_data['Unique Words % (Idea Density)'] * 100
# nice_stats_data.loc['MIN'] = nice_stats_data.min()
nice_stats_data.loc['Average'] = nice_stats_data.mean()
# nice_stats_data.loc['MAX'] = nice_stats_data.max()
nice_stats_data

In [31]:
# nice_stats_html = nice_stats_data.astype('int').to_html(classes='')

In [ ]:
# Beautify
(
    nice_stats_data
    .transpose()
    .astype('int')
    .style
    # .format("{:.2f}")

    # .bar(axis=1, cmap='viridis', width=100, height=80)

    # .background_gradient(cmap='viridis', axis=1)
    # .background_gradient(cmap='PiYG', axis=1)
    # .background_gradient(cmap='bwr_r', axis=1)
    # .background_gradient(cmap='magma_r', axis=1)
    # .background_gradient(cmap='Wistia', axis=1)
    # .background_gradient(cmap='cool', axis=1)  # 80s
    # .background_gradient(cmap='pink_r', axis=1)
    .background_gradient(cmap='Blues', axis=1)  # Like like
    # .background_gradient(cmap='GnBu', axis=1)  # Like like
    # .background_gradient(cmap='PuBuGn', axis=1)  # Like
    # .background_gradient(cmap='BuGn', axis=1)  #Like
    # .background_gradient(cmap='YlGn', axis=1)  # Like
    # .background_gradient(cmap='Oranges', axis=1)  # Like
    # .background_gradient(cmap='PuRd', axis=1)  # Like
    # .background_gradient(cmap='bone_r', axis=1)  # Like
    
    # .highlight_max(axis=1, color='pink').highlight_min(axis=1, color='powderblue')
)

## Bar Plots

In [33]:
import colorsys

import seaborn as sns

import lib.helper_functions as hf

# Select 6 colors from a palette
_colors = sns.color_palette('hsv', len(nice_stats_data.index[:-1]))

# This is how sns.color_palette() works internally
# import matplotlib.pyplot as plt
# import numpy as np
# plt.cm.viridis( np.linspace(0, 1, len(nice_stats_data.index[:-1]) + 2)[1:-1] )

# Convert RGB colors to HLS color space
_colors_hls = [colorsys.rgb_to_hls(*c) for c in _colors]
# Adjust lightness and saturation values
_colors_hls = [(h, l * 1.6, s * 0.75) for h, l, s in _colors_hls]
# Convert back to RGB color space
_colors = [colorsys.hls_to_rgb(*c) for c in _colors_hls]

color_by_name = {name: _colors[i] for i, name in enumerate(nice_stats_data.index[:-1])}

In [ ]:
figs_barplot = {}
for _colname in nice_stats_data.columns:
    _sorted_by_value_names = (
        nice_stats_data[[_colname]].iloc[0:-1].sort_values(by=_colname, ascending=False).index
    )

    figs_barplot[_colname] = hf.barplot(
        data=nice_stats_data[[_colname]]
        .iloc[0:-1]
        .sort_values(by=_colname, ascending=False)
        .transpose(),
        title=_colname,
        # xlabel='Speaker',
        # ylabel='Action Items',
        # show_mean=False,
        # show_mean_value=False,
        # show_values=True,
        colors=[color_by_name[name] for name in _sorted_by_value_names],
    )

## Action pie chart

In [ ]:
df_speaker_stats['action_items_count'].index.to_list(), df_speaker_stats['action_items_count'].to_list()

In [ ]:
import matplotlib.pyplot as plt

# labels = 'Frogs', 'Hogs', 'Dogs', 'Logs'
# sizes = [15, 30, 45, 10]
# fig, ax = plt.subplots()
# ax.pie(sizes, labels=labels)

fig_action_items, ax = plt.subplots()
colors = plt.get_cmap('Set3').colors
_df_speaker_stats = df_speaker_stats['action_items_count'].sort_index()
ax.pie(
    _df_speaker_stats.to_list(),
    labels=_df_speaker_stats.index.to_list(),
    autopct=lambda x: round(x * _df_speaker_stats.sum() / 100),
    pctdistance=0.8,
    colors=colors,
    counterclock=False,
    startangle=90,
);
_df_speaker_stats

## Sentiment evolution

### Overall Sentiment evolution

In [ ]:
df_sentiment['sentiment'].value_counts(normalize=True)

In [ ]:
import matplotlib.pyplot as plt
# df_sentiment.set_index('start')['sentiment_value'].plot(alpha=0.2)

_window_size_pct = 0.1
_window_size = int(
    df_sentiment.set_index('start')['sentiment_value'].shape[0] * _window_size_pct
)  # Window size as percentage
df_sentiment_mean = df_sentiment.set_index('start')['sentiment_value'].rolling(window=_window_size, center=True, min_periods=1).mean()
# ax = df_sentiment.set_index('start')['sentiment_value'].plot()
ax = df_sentiment_mean.plot(figsize=(10, 2), alpha=0)
ax.set_ylim(-1, 1)
# ax.axhline(y=0, color='lightgray', linestyle='--', linewidth=1)
ax.fill_between(
    df_sentiment_mean.index,
    df_sentiment_mean,
    0,
    where=(df_sentiment_mean > 0),
    color='lime',
    alpha=1,
)
ax.fill_between(
    df_sentiment_mean.index,
    df_sentiment_mean,
    0,
    where=(df_sentiment_mean < 0),
    color='deeppink',
    alpha=1,
)
ax.set_yticks([-1, 0, 1])
# ax.set_yticklabels(['Negative', 'Neutral', 'Positive'])
ax.set_yticklabels(['-', '0', '+'])
ax.set_xticks([])
ax.set_xlabel('')
ax.set_title('Sentiment Evolution')
plt.tight_layout()
plt.autoscale(tight=True, axis='x')
# df_sentiment.set_index('start')['sentiment_value'].plot()
fig_convo_sentiment_evo = ax.figure

In [ ]:
ax

### Sentiment evolution per person

In [ ]:
df_sentiment[['name','sentiment_value']].describe()#.groupby('name').std()

In [ ]:
df_sentiment.groupby('name')['sentiment_value'].apply(
    lambda x: pd.DataFrame((x.sum(),x.std(),x.mean()), index=('sum','std','mean'))
).unstack().droplevel(0, axis=1).sort_values('mean').style.background_gradient(cmap='RdPu', axis=1)

In [ ]:
import matplotlib.pyplot as plt
import math

pltlogger = logging.getLogger('matplotlib')
pltlogger.setLevel(logging.WARNING)

ncols = 2
nrows = math.ceil(df_speaker_stats['speaker'].shape[0] / ncols)
fig_speaker_sentiment_evo, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, nrows * 2))
# Don't show axis for not plotted positions
noshow_count = nrows * ncols - df_speaker_stats['speaker'].shape[0]
for i in range(nrows * ncols - 1, nrows * ncols - 1 - noshow_count, -1):
    axes[i // ncols, i % ncols].axis("off")  # Don't show

# for name in df_sentiment['name'].unique():
for i, name in enumerate(sorted(df_sentiment['name'].unique())):
    _df_sentiment_by_person = (
        df_sentiment.set_index('start')
        .query(f'name == "{name}"')['sentiment_value']
        .reindex(df_sentiment.set_index('start').index, fill_value=0)
    )

    # _df_sentiment_by_person.reset_index(drop=True).plot(
    #     alpha=0.2,
    #     )

    # Window size as percentage
    _window_size_pct = 0.10
    _window_size_computed = int(_df_sentiment_by_person.shape[0] * _window_size_pct)
    _window_size = max(1, _window_size_computed)

    df_sentiment_mean = (
        _df_sentiment_by_person
        # .reset_index(drop=True)
        .rolling(window=_window_size, center=True, min_periods=1).mean()
    )
    print(name, df_sentiment_mean.shape)
    if len(axes.shape) == 1:
        ax = df_sentiment_mean.plot(ax=axes[i % ncols], alpha=0)
    else:
        ax = df_sentiment_mean.plot(ax=axes[i // ncols, i % ncols], alpha=0)
    ax.fill_between(
        df_sentiment_mean.index,
        df_sentiment_mean,
        0,
        where=(df_sentiment_mean > 0),
        color='lime',
        alpha=1,
    )
    ax.fill_between(
        df_sentiment_mean.index,
        df_sentiment_mean,
        0,
        where=(df_sentiment_mean < 0),
        color='deeppink',
        alpha=1,
    )
    ax.set_yticks([-1, 0, 1])
    # ax.set_yticklabels(['Negative', 'Neutral', 'Positive'])
    ax.set_yticklabels(['-', '0', '+'])
    ax.set_xticks([])
    ax.set_xlabel('')
    ax.set_title(f'Sentiment Evolution ({name})')
    ax.autoscale(tight=True, axis='x')
    # break
    # plt.show()
fig_speaker_sentiment_evo.tight_layout()

## Words per minute

### Words per minute evolution (Excitement proxy)

In [ ]:
_window_size_pct = 0.1
_window_size = int(
    df_utter.set_index('start')['words_per_min'].shape[0] * _window_size_pct
)  # Window size as percentage

ax = (
    df_utter.set_index('start')['words_per_min']
    .rolling(window=_window_size, min_periods=1)
    .mean()
    .plot(figsize=(10, 2), label='')
)
ax.set_title('Words Per Minute Evolution')
ax.set_xticks([])
ax.set_xlabel('')
ax.set_ylim(0, 300)
ax.autoscale(tight=True, axis='x')
ax.grid(axis='y', alpha=0.3)
_mean = df_utter.set_index('start')['words_per_min'].mean()
ax.axhline(y=_mean, color='lime', zorder=-1, label=f'Average = {round(_mean,1)}')
ax.legend()
plt.tight_layout()
plt.autoscale(tight=True, axis='x')

fig_convo_wpm = ax.figure

### Words per minute evolution (Excitement per person proxy)

In [ ]:
import matplotlib.pyplot as plt

pltlogger = logging.getLogger('matplotlib')
pltlogger.setLevel(logging.WARNING)

ncols = 2
nrows = math.ceil(df_speaker_stats['speaker'].shape[0] / ncols)
fig_speaker_wpm, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, nrows * 2))
# Don't show axis for not plotted positions
noshow_count = nrows * ncols - df_speaker_stats['speaker'].shape[0]
for i in range(nrows * ncols - 1, nrows * ncols - 1 - noshow_count, -1):
    axes[i // ncols, i % ncols].axis("off")  # Don't show

# for name in df_utter['name'].unique():
for i, name in enumerate(sorted(df_sentiment['name'].unique())):

    _df_words_per_min_by_person = (
        df_utter.set_index('start')
        .query(f'name == "{name}"')['words_per_min']
        .reindex(df_utter.set_index('start').index)
    )
    # _df_sentiment_by_person = df_sentiment.set_index('start').query(f'name == "{name}"')['sentiment_value'].reindex(df_sentiment.set_index('start').index, fill_value=0)

    # (
    #     _df_words_per_min_by_person
    #     # .reset_index(drop=True)
    #     .plot(
    #         # marker='+',
    #         # linestyle='none',
    #         alpha=0.2,
    #     )
    # )

    # Window size as percentage
    _window_size_pct = 0.10
    _window_size_computed = int(_df_words_per_min_by_person.shape[0] * _window_size_pct)
    _window_size = max(1, _window_size_computed)

    df_words_per_min_mean = (
        _df_words_per_min_by_person
        # .reset_index(drop=True)
        .rolling(window=_window_size, center=True, min_periods=1)
        .mean()
    )
    if len(axes.shape) == 1:
        ax = df_words_per_min_mean.plot(label='', ax=axes[i % ncols])
    else:
        ax = df_words_per_min_mean.plot(label='', ax=axes[i // ncols, i % ncols])
    ax.set_title(f'Words per minute Evolution ({name})')
    ax.set_ylim(0, 400)
    ax.set_xticks([])
    ax.set_xlabel('')
    ax.autoscale(tight=True, axis='x')
    ax.grid(axis='y', alpha=0.3)
    _mean = _df_words_per_min_by_person.mean()
    ax.axhline(y=_mean, color='lime', zorder=-1, label=f'Average = {round(_mean,1)}')
    ax.legend()
fig_speaker_wpm.tight_layout()

## Chord diagram

In [ ]:
# Chord interactions with previous person

if not path_reports_dir.joinpath('chord.png').is_file():
    from plotapi import Chord

    Chord.api_key(os.getenv('PLOTAPI_KEY'))

    _df_interaction = df_interactions_prev.iloc[:-1, :-1]

    chord_interactions_prev = Chord(
        matrix=_df_interaction.values.tolist(),
        names=_df_interaction.index.to_list(),
        padding=0.5,
        curved_labels=True,
        conjunction="&",
        verb="interacted",
        noun="times",
        popup_width=100,
    )

    chord_interactions_prev.to_png(filename=path_reports_dir.joinpath('chord.png'))

from IPython.display import Image
Image(path_reports_dir.joinpath('chord.png'), width=500)

In [46]:
# png = chord.get_png()
# buf_chord = buf = io.BytesIO(png)
# image_chord = Image.open(buf_chord)
# # buf_chord.seek(0)  # Important: go to the start of the buffer


## Key Contributions

In [ ]:
import textwrap
for speaker in speaker_map:
    print('\n\n', speaker_map[speaker]['name'])
    for quote in speaker_map[speaker]['personal_key_contributions']:
        print('*', quote['tagline'])
        print(textwrap.fill(quote['quote'], width=100))

## Summary by participant

In [ ]:
import textwrap

for speaker in speaker_map:
    print('\n\n', speaker_map[speaker]['name'])
    print(textwrap.fill(speaker_map[speaker]['personal_summary'], width=100)) # Remove `textwrap.fill` to not have line breaks
    # print(speaker_map[speaker]['personal_summary'])

## Action items

In [ ]:
import textwrap
for speaker in speaker_map:
    print('\n\n', speaker_map[speaker]['name'])
    for action in speaker_map[speaker]['personal_action_items']:
        print('*', action['title'])
        print(textwrap.fill(action['action'], width=100))

# PDF generation

In [ ]:
_df = df_interactions_next.iloc[:-1, :-1]  # We remove one to avoid the All column or row
size = _df.shape[0]
for i in range(size):
    # print(_df.iat[i, i])
    _df.iat[i, i] = pd.NA
_list = _df.melt()['value'].dropna().tolist()
gini_reverse = round(100 * (1 - hf.gini(_list)), 2)


import lib.pdf_functions as pdffn
import lib.helper_functions as hf
import traceback
from urllib.parse import quote_plus

try:
    pdffn.create_report(
        images_barplot=[hf.figure_to_image(figs_barplot[fig]) for fig in figs_barplot],
        image_convo_sentiment_evo=hf.figure_to_image(fig_convo_sentiment_evo),
        image_speaker_sentiment_evo=hf.figure_to_image(fig_speaker_sentiment_evo),
        image_convo_wpm=hf.figure_to_image(fig_convo_wpm),
        image_speaker_wpm=hf.figure_to_image(fig_speaker_wpm),
        image_action_items=hf.figure_to_image(fig_action_items),
        path_image_chord=path_reports_dir.joinpath('chord.png'),
        path_report_assets=path_report_assets,
        summaries=summaries,
        gini_reverse_interventions=gini_reverse_interventions,
        gini_reverse_interactions=gini_reverse_interactions,
        gini_reverse_interactions_both_wrd_cnt=gini_reverse_interactions_both_wrd_cnt,
        nice_stats_data=nice_stats_data.transpose(),
        speaker_map=speaker_map,
        path_file=path_reports_dir.joinpath(f'{file_name_noext} - report - group.pdf'),
        link_feedback=(
            'https://docs.google.com/forms/d/e/'
            + '1FAIpQLSfgKdmjEgNkyyLIWIzjUiNrPHShUzLu4q8g7xXyNldzwRDKog'
            + '/viewform?entry.1276450318='
            + quote_plus(file_name_noext)
        ),
    )
except Exception as e:
    print(e)
    traceback.print_exc()
    raise e

# IDEAS

## Visualize Chapters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

chap_times = np.ndarray(0)
texts = np.ndarray(0)

for chap in transcript['chapters']:
    # print(chap)
    start = chap['start']
    end = chap['end']
    duration = chap['end'] - chap['start']
    middle_point = int(start + duration / 2)
    # chap_times.append((start, end, end-start))
    chap_times = np.append(chap_times, (start, end, duration, middle_point))
    texts = np.append(texts, chap['headline'])

    # # Vertical bar
    # plt.text(0.5, -middle_point, chap['headline'])

# Unnecessary for plot, scale is computed automatically
# but transforms into seconds
chap_times = np.array(chap_times) / 1000

chap_times.shape = [len(transcript['chapters']), 4]
starts = chap_times[:, 0]
ends = chap_times[:, 1]
durations = chap_times[:, 2]
middle_points = chap_times[:, 3]

# See https://ultrapythonic.com/colormaps-in-matplotlib/
# colors = plt.get_cmap('tab20c').colors
# colors = plt.get_cmap('Pastel1').colors
colors = plt.get_cmap('Pastel2').colors
# colors = plt.get_cmap('Set3').colors

# Horizontal bar
# plt.figure(figsize=(10, 1))
# plt.ylim(-0.8, 0.8)
# plt.barh([0], durations, left=starts, height=1, color=colors)
# plt.annotate("", xytext=(0, 0.7), xy=(ends[-1]* (3/4), 0.7), arrowprops=dict(arrowstyle="->"))
# for mid, txt in zip(middle_points, texts):
#     plt.text(mid, 0.9, txt, size=8, rotation=45)

# # Vertical bar
plt.figure(figsize=(0.5, 9))
plt.xlim(-0.8, 0.8)
plt.bar([0], -durations, bottom=-starts, width=1, color=colors)
plt.annotate("", xytext=(0.7, 0), xy=(0.7, -ends[-1]* (3/4)), arrowprops=dict(arrowstyle="->"))
for mid, txt in zip(middle_points, texts):
    plt.text(0.9, -mid, txt, size=8, verticalalignment='center')

plt.axis('off');

## Find Errors and Propose corrections

In [52]:
speaker_map_name_to_id = {speaker_map[key]['name']: key for key in speaker_map.keys()}
# speaker_map_name_to_id

In [53]:
# from IPython.display import display, Markdown

# # print('summary_one_sentence word count:', len(summary_one_paragraph.split()))
# display(Markdown(speaker_map[speaker_map_name_to_id['Felipe']]['errors_corrections']))